In [0]:
%sql
DESCRIBE TABLE laddstolpar_df.bronze.trafa_t10026

In [0]:
%sql
SELECT regkom, ar,
       SUM(CASE WHEN drivmedel <> 't1' THEN CAST(itrfslut AS BIGINT) END) AS sum_fuels,
       MAX(CASE WHEN drivmedel =  't1' THEN CAST(itrfslut AS BIGINT) END) AS total_t1,
       MAX(CASE WHEN drivmedel = '105' THEN CAST(itrfslut AS BIGINT) END) AS phev
FROM laddstolpar_df.bronze.trafa_t10026
WHERE regkom IN ('0180', '2584') AND ar = '2025'
GROUP BY regkom, ar;

In [0]:
%sql DESCRIBE TABLE laddstolpar_df.bronze.elpris

In [0]:
%sql
SELECT COUNT(*)                               AS silver_rows,
       (SELECT COUNT(*) FROM laddstolpar_df.bronze.elpris) AS bronze_rows,
       MIN(time_start_utc), MAX(time_start_utc),
       SUM(CASE WHEN sek_per_kwh < 0 THEN 1 END) AS negative,
       MAX(length(split(CAST((SELECT MAX(SEK_per_kWh) FROM laddstolpar_df.bronze.elpris) AS STRING), '\\.')[1])) AS max_decimals_sample
FROM laddstolpar_df.silver.elpris;

In [0]:
%sql
SELECT elomrade, datum_lokal, time_start_utc, time_end_utc,
       unix_timestamp(time_end_utc) - unix_timestamp(time_start_utc) AS seconds,
       sek_per_kwh, _file
FROM laddstolpar_df.silver.elpris
WHERE unix_timestamp(time_end_utc) - unix_timestamp(time_start_utc) <> 900
   OR sek_per_kwh NOT BETWEEN -10 AND 50
ORDER BY time_start_utc, elomrade;

In [0]:
%sql
SELECT elomrade, time_start, time_end
FROM laddstolpar_df.bronze.elpris
WHERE elomrade = 'SE3'
  AND time_start BETWEEN '2025-10-26T02:30' AND '2025-10-26T03:15'
ORDER BY _file, time_start;

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(datum_lokal) AS with_date,
       MIN(datum_lokal), MAX(datum_lokal)
FROM laddstolpar_df.silver.elpris;

In [0]:
%sql
SELECT expected_rows, COUNT(*) AS zone_days,
       SUM(CASE WHEN n_rows = expected_rows THEN 1 ELSE 0 END) AS complete,
       SUM(CASE WHEN n_rows <> n_distinct_starts THEN 1 ELSE 0 END) AS with_duplicates
FROM laddstolpar_df.ops.elpris_day_check
GROUP BY expected_rows
ORDER BY expected_rows;

In [0]:
%sql
DESCRIBE TABLE laddstolpar_df.bronze.seed_skr_kommungrupp;
DESCRIBE TABLE laddstolpar_df.bronze.seed_elomrade_lan_default;
DESCRIBE TABLE laddstolpar_df.bronze.seed_elomrade_kommun_override;

In [0]:
%sql
-- 1. Shape: expect 290 / 290 / 79 override / 211 county default
SELECT COUNT(*) AS rows, COUNT(DISTINCT kommun_kod) AS kommuner,
       SUM(CASE WHEN elomrade_level = 'kommun_override' THEN 1 ELSE 0 END) AS override,
       SUM(CASE WHEN elomrade_level = 'lan_default'     THEN 1 ELSE 0 END) AS lan_default,
       SUM(CASE WHEN elomrade_is_split     THEN 1 ELSE 0 END) AS split,          -- expect 13
       SUM(CASE WHEN elomrade_needs_review THEN 1 ELSE 0 END) AS needs_review    -- expect 12
FROM laddstolpar_df.silver.kommun;

-- 2. Zone distribution: expect SE1 19, SE2 34, SE3 177, SE4 60 (bronze check 2026-09-24)
SELECT elomrade, COUNT(*) FROM laddstolpar_df.silver.kommun GROUP BY elomrade ORDER BY elomrade;

-- 3. Override codes that did NOT match an SKR kommun (a typo would be silently lost): expect 0 rows
SELECT o.kommun_kod, o.kommun_namn
FROM laddstolpar_df.bronze.seed_elomrade_kommun_override o
LEFT ANTI JOIN laddstolpar_df.silver.kommun k USING (kommun_kod);

-- 4. Name mismatch between seeds (same code, different spelling): expect 0 rows
SELECT k.kommun_kod, k.kommun_namn AS skr_namn, o.kommun_namn AS override_namn
FROM laddstolpar_df.silver.kommun k
JOIN laddstolpar_df.bronze.seed_elomrade_kommun_override o USING (kommun_kod)
WHERE k.kommun_namn <> o.kommun_namn;

In [0]:
%sql
SELECT _file, COUNT(*) AS rows, MAX(_ingested_at) AS loaded
FROM laddstolpar_df.bronze.seed_elomrade_kommun_override
GROUP BY _file;

In [0]:
%sql
-- a) The 12 reviewed kommuner, now with their sourced primary
SELECT kommun_kod, kommun_namn, elomrade AS primary_zone, elomrade_secondary, elomrade_basis
FROM laddstolpar_df.silver.kommun
WHERE elomrade_is_split
ORDER BY kommun_kod;

-- b) The zone distribution after the review (before: SE1 19, SE2 34, SE3 177, SE4 60)
SELECT elomrade, COUNT(*) FROM laddstolpar_df.silver.kommun GROUP BY elomrade ORDER BY elomrade;

In [0]:
%sql
SELECT DISTINCT _file FROM laddstolpar_df.bronze.seed_elomrade_kommun_override;